# 📊 Data Scaling with RobustScaler
In this notebook, we explore **RobustScaler** using `sampledata1.csv` (Superstore Sales dataset).

### 🔍 What is RobustScaler?
When a dataset contains **extreme outliers** (like unusually high Sales or negative Profits), standard scaling methods like `StandardScaler` (Mean & Std) or `MinMaxScaler` (Min & Max) get heavily distorted.

**RobustScaler** solves this by removing the median and scaling the data according to the **Interquartile Range (IQR)**:

$$\text{Scaled Value} = \frac{X - \text{Median}}{\text{IQR}}$$

Where:
- $\text{Median} = Q_2$ (50th percentile)
- $\text{IQR} = Q_3 - Q_1$ (75th percentile - 25th percentile)

Because it uses **Median** and **IQR**, it is **not influenced by extreme outliers**!

In [ ]:
# Step 1: Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('tab10')

print("All libraries imported successfully!")

In [ ]:
# Step 2: Load the dataset and inspect numerical columns
df = pd.read_csv('sampledata1.csv')

# The numerical columns in this dataset
num_cols = ['Sales', 'Quantity', 'Discount', 'Profit']

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nSummary Statistics BEFORE Scaling:")
display(df[num_cols].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']])

### 📉 Visualizing Outliers Before Scaling
Notice how `Sales` max is **~$22,638** while median is only **~$54**! And `Profit` has extreme negative values (**-$6,599**) and positive values (**+$8,399**).
Let's visualize the outliers using **Boxplots**:

In [ ]:
# Step 3: Boxplots of Original Data showing Outliers
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Original Data Boxplots (Before Scaling) - Extreme Outliers Present', fontsize=14, fontweight='bold')

for idx, col in enumerate(num_cols):
    ax = axes[idx // 2, idx % 2]
    sns.boxplot(x=df[col], ax=ax, color='#4A90E2', flierprops={'marker':'o', 'markersize':3, 'alpha':0.3})
    ax.set_title(f'Boxplot of {col}', fontsize=12, fontweight='bold')
    ax.set_xlabel(col)

plt.tight_layout()
plt.show()

### ⚙️ Step 4: Applying RobustScaler
Now we initialize `RobustScaler()` and scale `Sales`, `Quantity`, `Discount`, and `Profit`.

In [ ]:
# Step 4: Initialize and Apply RobustScaler
scaler = RobustScaler()

# Create scaled columns
df_scaled = df.copy()
scaled_col_names = [f'{col}_robust' for col in num_cols]
df_scaled[scaled_col_names] = scaler.fit_transform(df[num_cols])

print("Summary Statistics AFTER RobustScaler:")
display(df_scaled[scaled_col_names].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']])

### 📈 Step 5: Graphical Comparison (Before vs After Scaling)
Let's compare the distributions of `Sales` and `Profit` side-by-side using histograms and KDE curves.

In [ ]:
# Step 5: Side-by-side Distribution Comparison for Sales and Profit
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle('Distribution Comparison: Original vs RobustScaler', fontsize=15, fontweight='bold')

# 1. Sales Original
sns.histplot(df['Sales'], kde=True, ax=axes[0, 0], color='#E74C3C', bins=50)
axes[0, 0].set_title('Original Sales (Skewed by Outliers)', fontweight='bold')
axes[0, 0].set_xlim(-100, 2500)  # Zoom in to see main distribution

# 2. Sales RobustScaled
sns.histplot(df_scaled['Sales_robust'], kde=True, ax=axes[0, 1], color='#2ECC71', bins=50)
axes[0, 1].set_title('RobustScaled Sales (Centered at Median = 0)', fontweight='bold')
axes[0, 1].set_xlim(-2, 15)

# 3. Profit Original
sns.histplot(df['Profit'], kde=True, ax=axes[1, 0], color='#E74C3C', bins=50)
axes[1, 0].set_title('Original Profit (Extreme Negative & Positive Outliers)', fontweight='bold')
axes[1, 0].set_xlim(-500, 500)

# 4. Profit RobustScaled
sns.histplot(df_scaled['Profit_robust'], kde=True, ax=axes[1, 1], color='#2ECC71', bins=50)
axes[1, 1].set_title('RobustScaled Profit (Centered at Median = 0)', fontweight='bold')
axes[1, 1].set_xlim(-15, 15)

plt.tight_layout()
plt.show()

In [ ]:
# Step 6: 2D Relationship: Sales vs Profit (Before vs After)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Relationship: Sales vs Profit (Before vs After RobustScaler)', fontsize=15, fontweight='bold')

# Before
ax1.scatter(df['Sales'], df['Profit'], alpha=0.4, color='#3498DB', edgecolors='none', s=25)
ax1.set_title('Original: Sales vs Profit ($)', fontweight='bold')
ax1.set_xlabel('Sales ($)')
ax1.set_ylabel('Profit ($)')
ax1.axhline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.7)

# After
ax2.scatter(df_scaled['Sales_robust'], df_scaled['Profit_robust'], alpha=0.4, color='#9B59B6', edgecolors='none', s=25)
ax2.set_title('RobustScaled: Sales vs Profit (Normalized)', fontweight='bold')
ax2.set_xlabel('Sales (Robust Scaled)')
ax2.set_ylabel('Profit (Robust Scaled)')
ax2.axhline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.7)
ax2.axvline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.7)

plt.tight_layout()
plt.show()

### 💡 Key Takeaways:
1. **Median Centering:** Notice that the 50th percentile (Median) for all scaled features is now exactly **0.0**.
2. **Outlier Resistance:** Unlike `MinMaxScaler` which squashes 99% of your data into a tiny flat clump due to 1 big outlier, `RobustScaler` maintains the natural spread of normal data.
3. **IQR Scaling:** Values between $-1$ and $+1$ represent the core 50% of your data (the Interquartile Range).